# Génération de formes à partir des règles
 - Génération : génération des formes d'après les contextes phonologiques
 - Filtrage : extraction du paradigme
- Évaluation  

## Importations
- codecs pour les encodages
- pandas et numpy pour les calculs sur tableaux
- matplotlib pour les graphiques
- itertools pour les itérateurs sophistiqués (paires sur liste, ...)

In [89]:
# -*- coding: utf8 -*-
import codecs,operator,datetime,os,glob
import features
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools as it
import pickle
import networkx as nx
#%pylab inline
#pd.options.display.mpl_style = 'default'
debug=False
from __future__ import print_function

def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [90]:
%matplotlib inline
%store -r ordStemCells
# ordStemCells

In [91]:
import yaml

In [92]:
from IPython.display import display, HTML

In [93]:
import datetime
def dateheure():
    return datetime.datetime.utcnow().strftime('%y%m%d%H%M')

In [94]:
saut="\n"

### Préparation des matrices de traits

In [95]:
features.add_config('/Users/gilles/Github/SWIM/ParadigmGeneration/Vlexique2/bdlexique.ini')
fs=features.FeatureSystem('phonemes')

# Choix de l'échantillon et des règles
- *sampleFile* est le nom de l'échantillon de départ
- *rulesFile* est le nom du fichier de règles
- *goldFile* est le nom du lexique Gold de référence

In [159]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
sampleFile="vlexique2-S4-omp.csv"
rulesFile="vlexique2-S4-omp-Regles.pkl"
fSample=repFiles+sampleFile
fInitial=fSample
fRules=repFiles+rulesFile

In [135]:
phonologicalMap="-X"
if "omp" in sampleFile:
    casesType="-Morphomes"
else:
    casesType=""
listeFormesOutput=["FS","FP"]
print(casesType)

-Morphomes


In [136]:
genDigraphe=False
genGraphe=False
genFormeVotes=True
genCliques=True
plotDistributionCliques=False

# Préparation du calcul des analogies

### Calcul de la différence entre deux formes

In [137]:
def diff(mot1,mot2):
    result=[]
    diff1=""
    diff2=""
    same=""
    vide="."
    lmax=max(len(mot1),len(mot2))
    lmin=min(len(mot1),len(mot2))
    for index in range(lmax):
        if index < lmin:
            if mot1[index]!=mot2[index]:
                diff1+=mot1[index]
                diff2+=mot2[index]
                same+=vide
            else:
                same+=mot1[index]
                diff1+=vide
                diff2+=vide
        elif index < len(mot1):
            diff1+=mot1[index]
        elif index < len(mot2):
            diff2+=mot2[index]
    diff1=diff1.lstrip(".")
    diff2=diff2.lstrip(".")
#    return (same,diff1,diff2,diff1+"_"+diff2)
    return (diff1+"-"+diff2)

### Accumulation des paires appartenant à un patron

In [138]:
def rowDiff(row, patrons):
    result=diff(row[0],row[1])
    if not result in patrons:
        patrons[result]=(formesPatron(),formesPatron())
    patrons[result][0].ajouterFormes(row[0])
    patrons[result][1].ajouterFormes(row[1])
    return (result[0],result[1])

### Transformation d'un patron en RegExp

In [139]:
def patron2regexp(morceaux):
    result="^"
    for morceau in morceaux:
        if morceau=="*":
            result+="(.*)"
        elif len(morceau)>1:
            result+="(["+morceau+"])"
        else:
            result+=morceau
    result+="$"
    result=result.replace(")(","")
    return result

### Substitution de sortie 
???

In [140]:
def remplacementSortie(sortie):
    n=1
    nsortie=""
    for lettre in sortie:
        if lettre==".":
            nsortie+="\g<%d>"%n
            n+=1
        else:
            nsortie+=lettre
    return nsortie

# Classe pour la gestion des patrons, des classes et des transformations

In [141]:
class paireClasses:
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classes1=classesPaire(case1,case2)
        self.classes2=classesPaire(case2,case1)

    def ajouterPatron(self,n,patron,motif):
        if n==1:
            self.classes1.ajouterPatron(patron,motif)
        elif n==2:
            self.classes2.ajouterPatron(patron,motif)
        else:
            if debug: print ("le numéro de forme n'est pas dans [1,2]",n, file=logfile)

    def ajouterPaire(self,forme1,forme2):
        self.classes1.ajouterPaire(forme1,forme2)
        self.classes2.ajouterPaire(forme2,forme1)
        
    def calculerClasses(self):
        return(self.classes1,self.classes2)

    
class classesPaire:
    '''
    Gestion des patrons, des classes et des transformations
    
    ajouterPatron : ajoute un patron et son motif associé (MGL)
    ajouterPaire : ajoute une paire de formes, calcule la classe de la forme1 et la règle sélectionnée
    sortirForme : cacule les formes de sortie correspondant à la forme1 avec leurs coefficients respectifs
    '''
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classe={}
        self.nbClasse={}
        self.patrons={}
        self.entree={}
        self.sortie={}
        self.classeCF={}
        self.nbClasseCF={}
    
    def ajouterPatron(self,patron,motif):
        self.patrons[patron]=motif
        (entree,sortie)=patron.split("-")
        self.entree[patron]=entree.replace(u".",u"(.)")
        self.sortie[patron]=remplacementSortie(sortie)
    
    def ajouterPaire(self,forme1,forme2):
        '''
        on calcule la classe de la paire idClasseForme et la règle sélectionnée
        on incrémente le compteur de la classe et celui de la règle sélectionnée à l'intérieur de la classe
        '''
        classeFormeCF=[]
        regleFormeCF=""
        classeForme=[]
        regleForme=""
        for patron in self.patrons:
            filterF1=".*"+patron.split("-")[0]+"$"
            if re.match(filterF1,forme1):
                classeFormeCF.append(patron)
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleFormeCF=patron
            filterF1=self.patrons[patron]
            if re.match(filterF1,forme1):
                classeForme.append(patron)
                '''
                le +"$" permet de forcer l'alignement à droite pour les transformations suffixales
                '''
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleForme=patron
        idClasseFormeCF=", ".join(classeFormeCF)
        if not idClasseFormeCF in self.classeCF:
            self.classeCF[idClasseFormeCF]={}
            self.nbClasseCF[idClasseFormeCF]=0
        if not regleFormeCF in self.classeCF[idClasseFormeCF]:
            self.classeCF[idClasseFormeCF][regleFormeCF]=0
        self.nbClasseCF[idClasseFormeCF]+=1
        self.classeCF[idClasseFormeCF][regleFormeCF]+=1
        
        idClasseForme=", ".join(classeForme)
        if not idClasseForme in self.classe:
            self.classe[idClasseForme]={}
            self.nbClasse[idClasseForme]=0
        if not regleForme in self.classe[idClasseForme]:
            self.classe[idClasseForme][regleForme]=0
        self.nbClasse[idClasseForme]+=1
        self.classe[idClasseForme][regleForme]+=1

    def sortirForme(self,forme,contextFree=True):
        classeForme=[]
        sortieForme={}
        for patron in self.patrons:
            if contextFree:
                filterF1=".*"+patron.split("-")[0]+"$"
            else:
                filterF1=self.patrons[patron]
            if re.match(filterF1,forme):
                classeForme.append(patron)
        if classeForme:
            idClasseForme=", ".join(classeForme)
            if contextFree:
                nbClasse=self.nbClasseCF
                classe=self.classeCF
            else:
                nbClasse=self.nbClasse
                classe=self.classe
            if idClasseForme in nbClasse:
                nTotal=nbClasse[idClasseForme]
                for patron in classe[idClasseForme]:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(classe[idClasseForme][patron])/nTotal
            else:
                if debug:
                    print (forme, file=logfile)
                    print ("pas de classe",idClasseForme, file=logfile)
                    print ("%.2f par forme de sortie" % (float(1)/len(classeForme)), file=logfile)
                nTotal=len(classeForme)
                for patron in classeForme:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(1)/nTotal
        else:
            if debug:
                print (forme, file=logfile) 
                print ("pas de patron", file=logfile)
        return sortieForme
        

## Appliquer la formule de calcul des différences entre chaines à chaque ligne

>si il y a au moins une ligne

>>on applique la différence à la ligne

>>on calcule les deux patrons par suppression des points initiaux

>>on renvoie le groupement par patrons (1&2)

>sinon

>>on renvoie le paradigme vide d'origine

In [142]:
def rapports(paradigme):
    if len(paradigme.columns.values.tolist())==2:
        (case1,lexeme)= paradigme.columns.values.tolist()
        case2=case1
    else:
        (case1,case2,lexeme)= paradigme.columns.values.tolist()
    patrons=pairePatrons(case1,case2)
    classes=paireClasses(case1,case2)
    if len(paradigme)>0:
        paradigme.apply(lambda x: patrons.ajouterFormes(x[case1],x[case2],diff(x[case1],x[case2])), axis=1)
        (regles1,regles2)=patrons.calculerGM()
        for regle in regles1:
            classes.ajouterPatron(1,regle,regles1[regle])
        for regle in regles2:
            classes.ajouterPatron(2,regle,regles2[regle])
        paradigme.apply(lambda x: classes.ajouterPaire(x[case1],x[case2]), axis=1)
    (classes1,classes2)=classes.calculerClasses()
    return (classes1,classes2)

### Dédoubler les lignes avec des surabondances dans *colonne*
>identifier une ligne avec surabondance

>>ajouter les lignes correspondant à chaque valeur

>>ajouter le numéro de la ligne initiale dans les lignes à supprimer

>supprimer les lignes avec surabondance

NB : il faut préparer le tableau pour avoir une indexation qui permette l'ajout des valeurs individuelles et la suppression des lignes de surabondances

In [143]:
def splitCellMates(df,colonne):
    '''
    Calcul d'une dataframe sans surabondance par dédoublement des valeurs
    '''
    test=df.reset_index()
    del test["index"]
    splitIndexes=[]
    for index,ligne in test.iterrows():
        if "," in ligne[colonne]:
            valeurs=set(ligne[colonne].split(","))
            nouvelleLigne=ligne
            for valeur in valeurs:
                nouvelleLigne[colonne]=valeur
                test=test.append(nouvelleLigne,ignore_index=True)
            splitIndexes.append(index)
    if splitIndexes:
        test=test.drop(test.index[splitIndexes])
    return test


# Lecture de l'échantillon

In [144]:
neutralisationsNORD=(u"6û",u"9ê")
neutralisationsSUD=(u"e2o",u"E9O")
if phonologicalMap=="-N":
    neutralisations=neutralisationsNORD
elif phonologicalMap=="-S":
    neutralisations=neutralisationsSUD
else:
    neutralisations=(u"",u"")
    phonologicalMap=("-X")
bdlexiqueIn = u"èò"+neutralisations[0]
bdlexiqueNum = [ord(char) for char in bdlexiqueIn]
neutreOut = u"EO"+neutralisations[1]
neutralise = dict(zip(bdlexiqueNum, neutreOut))

neutralisationsTotales=(u"e2o6û",u"E9O9ê")
totalNeutreIn=u"èò"+neutralisationsTotales[0]
totalNeutreNum=[ord(char) for char in totalNeutreIn]
totalNeutreOut=u"EO"+neutralisationsTotales[1]
totalNeutralise = dict(zip(totalNeutreNum, totalNeutreOut))

In [145]:
def recoder(chaine,table=totalNeutralise):
    if type(chaine)==str:
        temp=chaine.translate(table)
        result=temp
    elif type(chaine)==unicode:
        result=chaine.translate(table)
    else:
        result=chaine
    return result

### Vérification de la phonotactique des glides du français
- si *prononciation* est *None* renvoyer *None*
- ajout de diérèses dans les séquences mal-formées
- vérification des séquences consonne+glide à la finale

In [146]:
dierese={"j":"ij", "w":"uw","H":"yH","i":"ij","u":"uw","y":"yH"}

In [147]:
def checkFrench(prononciation):
    if prononciation and not pd.isnull(prononciation):
        result=recoder(prononciation)
        m=re.match(r"^.*([^ieèEaOouy926êôâ])[jwH]$",result)
        if m:
            print ("pb avec un glide final", [prononciation])
        m=re.match(r"(.*[ptkbdgfsSvzZ][rl])([jwH])(.*)",result)
        if m:
            n=re.search(r"[ptkbdgfsSvzZ][rl](wa|Hi|wê)",result)
            if not n:
                glide=m.group(2)
                result=m.group(1)+dierese[glide]+m.group(3)
        m=re.match(r"(.*)([iuy])([ieEaOouy].*)",result)
        if m:
            glide=m.group(2)
            result=m.group(1)+dierese[glide]+m.group(3)
    else:
        result=prononciation
    return result

In [148]:
paradigmes=pd.read_csv(fSample,sep=";",encoding="utf8")
if u"Unnamed: 0" in paradigmes.columns:
    del paradigmes[u"Unnamed: 0"]
paradigmes=paradigmes.dropna(axis=1,how='all')

In [149]:
paradigmes.columns

Index(['lexeme', 'ai1P', 'ai1S', 'ai2P', 'ai2S', 'ai3P', 'ai3S', 'fi1P',
       'fi2P', 'fi3P', 'ii1P', 'ii2P', 'inf', 'is1P', 'is1S', 'is2P', 'is3P',
       'is3S', 'pI1P', 'pI2P', 'pI2S', 'pP', 'pc1P', 'pc2P', 'pi1P', 'pi1S',
       'pi2P', 'pi3P', 'ps1P', 'ps2P', 'pc3P', 'is2S', 'ii3S', 'ppFP', 'ppMS',
       'fi3S', 'pi3S'],
      dtype='object')

In [150]:
sampleCases=paradigmes.columns.values.tolist()
sampleCases.remove(u"lexeme")
# sampleCases
analyseCases=sampleCases

#Adapt all the forms to French phonology
for case in sampleCases:
    paradigmes[case]=paradigmes[case].apply(lambda x: checkFrench(x))

In [151]:
dictMorphomeCases={}
if casesType=="-Morphomes":
    with open(fSample.replace(".csv",".yaml"),"r") as inFile:
        dictMorphomeCases=yaml.safe_load(inFile)
dictMorphomeCases

{'fi3S': ['fi2S', 'fi3S'],
 'ii3S': ['ii3S', 'ii1S', 'ii3P', 'ii2S'],
 'is2S': ['is2S', 'ps1S', 'ps2S', 'ps3P', 'ps3S'],
 'pc3P': ['pc3P', 'pc2S', 'pc1S', 'fi1S', 'pc3S'],
 'pi3S': ['pi2S', 'pi3S'],
 'ppFP': ['ppFP', 'ppFS'],
 'ppMS': ['ppMS', 'ppMP']}

In [152]:
for case in sampleCases:
    if case not in dictMorphomeCases:
        dictMorphomeCases[case]=[case]
dictMorphomeCases

{'fi3S': ['fi2S', 'fi3S'],
 'ii3S': ['ii3S', 'ii1S', 'ii3P', 'ii2S'],
 'is2S': ['is2S', 'ps1S', 'ps2S', 'ps3P', 'ps3S'],
 'pc3P': ['pc3P', 'pc2S', 'pc1S', 'fi1S', 'pc3S'],
 'pi3S': ['pi2S', 'pi3S'],
 'ppFP': ['ppFP', 'ppFS'],
 'ppMS': ['ppMS', 'ppMP'],
 'ai1P': ['ai1P'],
 'ai1S': ['ai1S'],
 'ai2P': ['ai2P'],
 'ai2S': ['ai2S'],
 'ai3P': ['ai3P'],
 'ai3S': ['ai3S'],
 'fi1P': ['fi1P'],
 'fi2P': ['fi2P'],
 'fi3P': ['fi3P'],
 'ii1P': ['ii1P'],
 'ii2P': ['ii2P'],
 'inf': ['inf'],
 'is1P': ['is1P'],
 'is1S': ['is1S'],
 'is2P': ['is2P'],
 'is3P': ['is3P'],
 'is3S': ['is3S'],
 'pI1P': ['pI1P'],
 'pI2P': ['pI2P'],
 'pI2S': ['pI2S'],
 'pP': ['pP'],
 'pc1P': ['pc1P'],
 'pc2P': ['pc2P'],
 'pi1P': ['pi1P'],
 'pi1S': ['pi1S'],
 'pi2P': ['pi2P'],
 'pi3P': ['pi3P'],
 'ps1P': ['ps1P'],
 'ps2P': ['ps2P']}

- sampleCases pour la liste des cases effectivement représentées dans le corpus de départ 

In [153]:
countInput=paradigmes.stack().value_counts(dropna=True).sum()
print("nombre de formes de départ",countInput)

nombre de formes de départ 43534


In [154]:
countLexemes=len(paradigmes.dropna(thresh=1)["lexeme"])

## Préparation des données initiales et de référence

In [155]:
listeTest=paradigmes.dropna(thresh=1)["lexeme"].values.tolist()
# listeTest=[u"asseoir",u"balayer",u"amalgamer",u"avoir",u"manger"]
nbVerbes=len(listeTest)
print (nbVerbes)

4242


### Préparation des formes initiales

In [156]:
initialParadigmes=pd.read_csv(fInitial,sep=";",encoding="utf8")
if u"Unnamed: 0" in paradigmes.columns:
    del paradigmes[u"Unnamed: 0"]
initialParadigmes=initialParadigmes.dropna(axis=1,how='all')

In [157]:
initialForms=pd.melt(initialParadigmes[paradigmes["lexeme"].isin(listeTest)],id_vars=["lexeme"]).dropna()

initialForms["lexeme-case"]=initialForms["lexeme"]+"-"+initialForms["variable"]
initialForms.drop(labels=["lexeme","variable"],axis=1,inplace=True)

initialForms.set_index(["lexeme-case"],inplace=True)

initialFormsIndex=initialForms.index.tolist()

### Préparation des formes de référence

# Lecture des règles

In [160]:
with open(fRules, 'rb') as input:
    resultatsLecture = pickle.load(input)
resultatsLecture

{('ai1P', 'ai1P'): <__main__.classesPaire at 0x1764304d0>,
 ('ai1P', 'ai1S'): <__main__.classesPaire at 0x302158790>,
 ('ai1S', 'ai1P'): <__main__.classesPaire at 0x3026d53d0>,
 ('ai1P', 'ai2P'): <__main__.classesPaire at 0x3026d4d50>,
 ('ai2P', 'ai1P'): <__main__.classesPaire at 0x3026d4110>,
 ('ai1P', 'ai2S'): <__main__.classesPaire at 0x3026d4190>,
 ('ai2S', 'ai1P'): <__main__.classesPaire at 0x3026d44d0>,
 ('ai1P', 'ai3P'): <__main__.classesPaire at 0x3026d5fd0>,
 ('ai3P', 'ai1P'): <__main__.classesPaire at 0x3026d40d0>,
 ('ai1P', 'ai3S'): <__main__.classesPaire at 0x3026d7590>,
 ('ai3S', 'ai1P'): <__main__.classesPaire at 0x3026d5ad0>,
 ('ai1P', 'fi1P'): <__main__.classesPaire at 0x3026d6f50>,
 ('fi1P', 'ai1P'): <__main__.classesPaire at 0x3026db890>,
 ('ai1P', 'fi2P'): <__main__.classesPaire at 0x3026dae50>,
 ('fi2P', 'ai1P'): <__main__.classesPaire at 0x3026ce990>,
 ('ai1P', 'fi3P'): <__main__.classesPaire at 0x3026e6790>,
 ('fi3P', 'ai1P'): <__main__.classesPaire at 0x3026e6150

### Comparer les cases analysées avec l'ensemble de toutes les cases

# Préparations pour la génération des formes

In [161]:
class paradigmeDistribution:
    '''
    Gestion des distributions dans les cases du paradigme
    '''

    def __init__(self,lexeme):
        self.lexeme=lexeme
        self.formes={i:{} for i in analyseCases}

    def ajouterFormes(self,case,formes,coef=1.0):
        for forme in formes:
            if not forme in self.formes[case]:
                self.formes[case][forme]=0
            self.formes[case][forme]+=formes[forme]*coef
            
    def normaliserDistributions(self,caseListe=analyseCases):
        normalesDistributions={i:{} for i in caseListe}
        for case in caseListe:
            total=0
            for element in self.formes[case]:
                total+=self.formes[case][element]
            for element in self.formes[case]:
                normalesDistributions[case][element]=float(self.formes[case][element])/total
        return normalesDistributions
        

In [162]:
def generateForms(lexeme,contextFree=False):
    # pb avec l'encodage des noms de lexèmes
    candidats=paradigmeDistribution(lexeme)
    bLexParadigme=paradigmes["lexeme"].str.contains("^"+lexeme+"$")
    casesSamples=paradigmes[bLexParadigme].columns[paradigmes[bLexParadigme].notnull().iloc[0]].tolist()
    casesSamples.remove("lexeme")
    for caseDepart in casesSamples:
        formeDepart=paradigmes[bLexParadigme][caseDepart].iloc[0]
        if debug: print (caseDepart,formeDepart, file=logfile)
#        if formeDepart!="nan":
        for case in analyseCases:
            if debug: print (case, file=logfile)
            if not isinstance(resultatsLecture[(caseDepart, case)],str):
                if "," in formeDepart:
                    formesDepart=formeDepart.split(",")
                    coef=1.0/len(formesDepart)
                    for element in formesDepart:
                        candidats.ajouterFormes(case,resultatsLecture[(caseDepart, case)].sortirForme(element,contextFree),coef)
                else:
                    candidats.ajouterFormes(case,resultatsLecture[(caseDepart, case)].sortirForme(formeDepart,contextFree))
            else: 
                if debug: print ("str", resultatsLecture[(caseDepart, case)], file=logfile)
        if candidats.formes[caseDepart]=={} :
            candidats.ajouterFormes(caseDepart,{formeDepart:1.0})
    return candidats

In [163]:
def ajouterPoint(lexeme,forme,case,digraphe,graphe):
    pointName="%s-%s-%s"%(lexeme,forme,case)
#    if not pointName in digraphe.nodes():
    tam=case[:2]
    if tam=="in": tam="inf"
    digraphe.add_node(pointName, tam='"%s"'%tam)
    graphe.add_node(pointName, tam='"%s"'%tam)
    return pointName

def ajouterFleche(pointDepart,pointSortie,coef,digraphe,graphe):
    digraphe.add_edge(pointDepart,pointSortie,weight=float(coef))
    if digraphe.has_edge(pointSortie,pointDepart):
        # modif pour networkx v2
        # coefGraphe=float(digraphe.edge[pointSortie][pointDepart]["weight"]+coef)/2
        coefGraphe=float(digraphe.adj[pointSortie][pointDepart]["weight"]+coef)/2
        graphe.add_edge(pointDepart,pointSortie,weight=coefGraphe)

In [164]:
def generateParadigms(generation1,genDigraphe=True,contextFree=False):
    lexeme=generation1.lexeme
    distributionInitiale=generation1.normaliserDistributions()
    candidats=paradigmeDistribution(lexeme)
    digraphe=nx.DiGraph()
    graphe=nx.Graph()    
    for caseDepart in analyseCases:
        for formeDepart in distributionInitiale[caseDepart]:
            if formeDepart:
                pointDepart=ajouterPoint(lexeme,formeDepart,caseDepart,digraphe,graphe)
                coefDepart=distributionInitiale[caseDepart][formeDepart]
                if debug: print (caseDepart,formeDepart, file=logfile)
                for caseSortie in analyseCases:
                    distributionSortieBrute=resultatsLecture[(caseDepart, caseSortie)].sortirForme(formeDepart,contextFree)
                    if distributionSortieBrute:
                        if not genDigraphe:
#                            print ("brute",distributionSortieBrute)
                            distributionSortie={f:distributionSortieBrute[f] for f in distributionSortieBrute if f in distributionInitiale[caseSortie]}
                        else:
                            distributionSortie=distributionSortieBrute
#                        print ("filtre",distributionSortie)
#                        print (distributionInitiale[caseSortie])
                        if debug: print (caseSortie,distributionSortie,distributionInitiale[caseDepart], file=logfile)
                        candidats.ajouterFormes(caseSortie,distributionSortie,distributionInitiale[caseDepart][formeDepart])
                        for formeSortie in distributionSortie:
                            pointSortie=ajouterPoint(lexeme,formeSortie,caseSortie,digraphe,graphe)
                            coefSortie=distributionSortie[formeSortie]
                            ajouterFleche(pointDepart,pointSortie,float(coefDepart*coefSortie),digraphe,graphe)
    return (candidats,digraphe,graphe)

# Génération d'un jeu de formes

In [165]:
def generate(lexeme,genDigraphe=True,contextFree=False):
    # print (lexeme,end=", ")
    generation1=generateForms(lexeme,contextFree)
    return generation1

In [166]:
paradigmes.dropna(thresh=1).count().sum()-paradigmes.dropna(thresh=1)["lexeme"].count()

39292

In [167]:
def cliqueScore(clique,graph):
    score=0
    if len(clique)>1:
        for (depart,arrivee) in it.combinations_with_replacement(clique,2):
            score+=graph[depart][arrivee]["weight"]
    return score

In [168]:
def splitArrivee(arrivee):
    arriveeMorceaux=arrivee.split("-")
    if len(arriveeMorceaux)<3:
        print (arrivee,arriveeMorceaux)
    lexeme="-".join(arriveeMorceaux[:-2])
    formeArrivee=arriveeMorceaux[-2]
    caseArrivee=arriveeMorceaux[-1]
    return (lexeme,formeArrivee,caseArrivee)

# trouver tous les liens vers FS-* et FP-*
# regrouper par forme 
# calculer les proportions
# renvoyer les proportions par forme
# avec le nombre de forme à l'appui
def formeScore(forme,graph):
    scores={}
    scoresNormes={}
    # modif pour networkx v2
    # for depart in graph.edge edge=>adj:
    for depart in graph.adj:
        for arrivee in graph.adj[depart]:
            (lexeme, formeArrivee, caseArrivee)=splitArrivee(arrivee)
            if caseArrivee==forme:
#                print (depart, formeArrivee, graph.edge[depart][arrivee])
                if not formeArrivee in scores:
                    scores[formeArrivee]=0
                scores[formeArrivee]+=graph.adj[depart][arrivee]["weight"]
    totalArrivee=0
    for formeArrivee in scores:
        totalArrivee+=scores[formeArrivee]
    for formeArrivee in scores:
        scoresNormes[formeArrivee]=scores[formeArrivee]/totalArrivee
    return (scores,scoresNormes)
        

# Génération de paradigmes

In [169]:
def generateParadigms(contextFree=False):
    paradigmes={}
    for i,element in enumerate(listeTest):
        paradigmes[element]={}
        if (i%100)==0: print (i, dateheure()[-4:], int(100*float(i)/nbVerbes), end=", ")
        generation=generate(element,contextFree)
        pElement={}
        for k,v in generation.formes.items():
            maxDist=0
            maxF=""
            for f in v:
                if v[f]>maxDist:
                    maxDist=v[f]
                    maxF=f
            pElement[k]=maxF
        paradigmes[element]=pElement
    return pd.DataFrame(paradigmes).T

In [170]:
newParadigms=generateParadigms()
if casesType=="-Morphomes":
    for m,cListe in dictMorphomeCases.items():
        for c in cListe:
            if c!=m:
                newParadigms[c]=newParadigms[m]
newParadigms[ordStemCells].to_csv("test.csv")


0 1645 0, 100 1645 2, 200 1645 4, 300 1645 7, 400 1645 9, 500 1645 11, 600 1645 14, 700 1645 16, 800 1645 18, 900 1646 21, 1000 1646 23, 1100 1646 25, 1200 1646 28, 1300 1646 30, 1400 1646 33, 1500 1646 35, 1600 1646 37, 1700 1646 40, 1800 1646 42, 1900 1646 44, 2000 1646 47, 2100 1646 49, 2200 1646 51, 2300 1646 54, 2400 1646 56, 2500 1646 58, 2600 1646 61, 2700 1646 63, 2800 1646 66, 2900 1646 68, 3000 1646 70, 3100 1646 73, 3200 1647 75, 3300 1647 77, 3400 1647 80, 3500 1647 82, 3600 1647 84, 3700 1647 87, 3800 1647 89, 3900 1647 91, 4000 1647 94, 4100 1647 96, 4200 1647 99, 

# Reste

In [42]:
def dictPdRowForms(row):
    result={}
    for case in sampleCases:
        print (case,row[case].values[0])
    return result

def tableZero(case):
    if case in sampleCases:
        return u"Ø"
    else:
        return u"="

def makeTable(dictForms,title=""):
    tabular=[]
    labelTenseCode={"pi":"Present","ii":"Imperfective","ai":"Simple Past","fi":"Future",
                    "ps":"Subjunctive Pres.","is":"Subjunctive Imp.","pc":"Conditional","pI":"Imperative",
                    "inf":"Infinitive",
                    "ppMS":"Past Part. MS","ppMP":"Past Part. MP",
                    "ppFS":"Past Part. FS","ppMP":"Past Part. FP"
                   }
    def makeLine6(tenseCode):
        line=[]
        line.append(r"<th>%s</th>"%labelTenseCode[tenseCode])
        for person in [per+nb for nb in ["S","P"] for per in ["1","2","3"]]:
            case=tenseCode+person
            if (case in dictForms) and (not (type(dictForms[case]) == float and np.isnan(dictForms[case]))):
                line.append(r"<td>%s</td>"%(dictForms[case]))
            else:
                line.append(r"<td>%s</td>"%(tableZero(case)))
        return r"<tr>"+r"".join(line)+r"</tr>"

    def makeLine3(tenseCode):
        line=[]
        line.append(r"<th>%s</th>"%labelTenseCode[tenseCode])
        for person in [per+nb for nb in ["S","P"] for per in ["1","2","3"]]:
            if person in ["2S","1P","2P"]:
                case=tenseCode+person
                if case in dictForms and (not (type(dictForms[case]) == float and np.isnan(dictForms[case]))):
                    line.append(r"<td>%s</td>"%(dictForms[case]))
                else:
                    line.append(r"<td>%s</td>"%(tableZero(case)))
            else:
                line.append(r"<td>%s</td>"%(u"---"))
        return r"<tr>"+r"".join(line)+r"</tr>"
    
    def makeLineNF():
        line=[]
        line.append(r"<th>%s</th>"%"NF")
        for case in ["inf","pP","ppMS","ppMP","ppFS","ppFP"]:
            if case in dictForms and (not (type(dictForms[case]) == float and np.isnan(dictForms[case]))):
                line.append(r"<td>%s</td>"%(dictForms[case]))
            else:
                line.append(r"<td>%s</td>"%(tableZero(case)))
        return r"<tr>"+r"".join(line)+r"</tr>"
    
        
    top=[
        r"<table>",
        r"<caption style='caption-side:bottom;text-align:center'>",
        "Verbe : %s"%title,
        r"</caption>",
#        r"<tr><th/><th>1S</th><th>2S</th><th>3S</th><th>1P</th><th>2P</th><th>3P</th></tr>"
        r"<tr><th/><th>1SG</th><th>2SG</th><th>3SG</th><th>1PL</th><th>2PL</th><th>3PL</th></tr>"
        ]
    bottom=[
        r"</table>"
        ]
    tabular.append("\n".join(top))
    for tenseCode in ["pi","ii","fi","pc", "ps","ai", "is"]:
        tabular.append(makeLine6(tenseCode))
    tabular.append(makeLine3("pI"))
    tabular.append(makeLineNF())
    tabular.append("\n".join(bottom))
    return "\n".join(tabular)    

def diffParadigme(lexeme):
    outLen=lexemeMaxCliques[lexeme][0]
    bLexParadigme=paradigmes["lexeme"].str.contains("^"+lexeme+"$")
    inLen=paradigmes[bLexParadigme].notnull().sum(axis=1).values[0]-1
    if outLen>inLen:
        print (lexemeMaxCliques[lexeme][1])
        print (paradigmes[paradigmes["lexeme"]=="grandir"].values)
    return outLen-inLen
    

In [45]:
def extendParadigmes(contextParadigmes,extendMorphomes=False):
    lexemesParadigmeListe=[]
    for lexeme in contextParadigmes:
        if extendMorphomes:
            lexParadigmes=filledOutClique(contextParadigmes[lexeme])
        else:
            lexParadigmes=contextParadigmes[lexeme]
        if len(lexParadigmes)!=1:
            if debug:
                print ("LEXEME WITH A NON UNIQUE PARADIGM PB",len(lexParadigmes),lexeme)
                print (lexParadigmes)
        lexParadigme=lexParadigmes[0]
        for lexForme in lexParadigme:
            lexemesParadigmeListe.append(cutNodeName(lexForme))
    newForms=pd.DataFrame(lexemesParadigmeListe)
    newForms.columns=["lexeme","form","case"]
#    newParadigmes=newForms.pivot(index="lexeme", columns="case", values="form")
    newParadigmes=pd.pivot_table(newForms, values='form', index=['lexeme'], columns=['case'], aggfunc=lambda x: ",".join(x)).reset_index().reindex()
    for i in newParadigmes.itertuples():
#        print (i[0],i[1])
        lexeme=i[1]
        bLexParadigme=paradigmes["lexeme"].str.contains("^"+lexeme+"$")
        lexemeIndexes=paradigmes.lexeme[bLexParadigme].index.tolist()
        if lexemeIndexes:
            lexemeIndex=lexemeIndexes[0]
        else:
            print (i,lexeme,lexemeIndexes)
        bLexNewParadigme=newParadigmes["lexeme"].str.contains("^"+lexeme+"$")
        newParadigmes.loc[bLexNewParadigme,"index"]=int(lexemeIndex)
    newParadigmes.set_index("index",inplace=True)   
    return paradigmes.combine_first(newParadigmes)

## Préparation des paradigmes

In [50]:
paradigmesOriginaux=paradigmes.copy()
paradigmesSample=paradigmesOriginaux[paradigmesOriginaux["lexeme"].isin(listeTest)]

In [51]:
paradigmes=newParadigmes.copy()
paradigmesColumns=paradigmes.columns.tolist()
for c in sampleCases:
    if not c in paradigmesColumns:
        print (c)
        paradigmes[c]=np.NaN

In [55]:
print(sampleFile)
print("nombre de cases du paradigme",len(sampleCases))
print()
print("nombre de formes de départ\t",countInput)
print("nombre de formes de Swim1\t",countSwim1)
print("nombre de formes de Swim2\t",countSwim2)
print("nombre de formes attendues\t",countLexemes*len(sampleCases))


vlexique2-S2-omp.csv
nombre de cases du paradigme 42

nombre de formes de départ	 67880
nombre de formes de Swim1	 169501
nombre de formes de Swim2	 177863
nombre de formes attendues	 191562
